# Lobe and brain tissue segmentation

End-to-end workflow for **3D lobe segmentation** on multi-channel microscopy (Leica `.lif` or TIFF):

1. **Load** a scene and preview channels with `stackview`
2. **Preprocess** — per-channel rescale, blur, gamma/sigmoid, channel max-projection
3. **Segment lobes** — tiled MicroSAM with region filters on cross-sectional area and aspect ratio
4. **Derive brain mask** — binary mask from all positive lobe labels
5. **Save** lobe and brain label volumes as scene-specific TIFF
6. **Analyze** — region properties table (`RegionAnalyzer`)
7. **Visualize** — optional Napari viewer with measurements as label features

**Prerequisites:** MicroSAM embeddings at `embedding_path`, GPU recommended for segmentation.

In [ ]:
import copy
import logging

import numpy as np
import stackview

import vistiq
from vistiq.io import ImageLoader, ImageLoaderConfig, ImageWriter, ImageWriterConfig, unstack_image
from vistiq.preprocess import FuncProcessorConfig, PreprocessFlow, PreprocessFlowConfig, RescaleConfig
from vistiq.segment import (
    MicroSAMSegmenterConfig,
    RangeFilterConfig,
    RegionAnalyzer,
    RegionAnalyzerConfig,
    RegionFilterConfig,
    TiledSegmentationFlow,
    TiledSegmentationFlowConfig,
)
from vistiq.utils import ArrayIteratorConfig, check_device

## Setup

Log vistiq messages and report available PyTorch devices (CUDA / MPS / CPU).

In [ ]:
logger = logging.getLogger(vistiq.__name__)
logger.info(f"Available Torch accelerators: {check_device()}")

## Load image

Set `path` to a `.lif` or TIFF file and `scene_index` for multi-scene containers. Channel names are normalized (`Red` → `Dpn`, etc.) via `ImageLoaderConfig.rename_channel`.

In [ ]:
#path="/standard/vol191/siegristlab/Microsam_Segmentation/Conditional Split/control_24+48/DCP1/1_Dpn.tif"
#path="/standard/vol191/siegristlab/Microsam_Segmentation/24h/AkhGal4 x OR Susie/Scrib488 Dpn555 EdU 647/Raw files/Animal 1.lif"
path="Animal 1.lif"
#path="/Users/khs3z/Documents/SDS_/projects/Siegrist/Microsam_Segmentation/24h/AkhGal4 x OR Susie/Scrib488 Dpn555 EdU 647/Animal 1.lif"

scene_index = 0

embedding_path = "./embeddings"
#embedding_path = "/standard/vol191/siegristlab/Sagar/microsam/embeddings/"

In [ ]:
ilc = ImageLoaderConfig(
    squeeze=True, 
    rename_channel={"Red": "Dpn", "Green": "Scrib", "Blue": "EdU"}, 
    scene_index=scene_index, 
    split_channels=False,
    #substack="C:3"
)
img, metadata = ImageLoader(ilc).run(path)

In [ ]:
if "C" in metadata["axes"]:
    vimg = np.concatenate(np.unstack(img, axis=0), axis=-1)
else:
    vimg = img
stackview.slice(vimg)
#stackview.switch(img, colormap=["pure_green", "pure_blue", "pure_red"], toggleable=True)

## Preprocess

Channel-wise pipeline on the loaded stack:

| Step | Purpose |
|------|---------|
| `Rescale` (2–98%) | Contrast stretch per channel |
| Gaussian blur | Noise reduction (`sigma=1`) |
| Gamma + sigmoid | Brighten dim structures |
| `Rescale` → `uint8` | Re-normalize before projection |
| `numpy.max` over `C` | Single-channel max-projection for segmentation |

`slice_def` on each step controls which axes are iterated (see comments in the cell).

In [ ]:
ppcfg = PreprocessFlowConfig(
    processors = [
        RescaleConfig(
            low=2, 
            high=98, 
            dtype=np.uint8, 
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1)) # ZYX over each channel
        ),
        FuncProcessorConfig(
            func="skimage.filters.gaussian",
            kwargs={"sigma": 1.0},
            iterator_config=ArrayIteratorConfig(slice_def=(-2,-1)) # YX over each Z-plane and channel
        ),
        FuncProcessorConfig(
            func="skimage.exposure.adjust_gamma",
            kwargs={"gamma": 0.2},
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1)) # ZYX over each channel
        ),
        FuncProcessorConfig(
            func="skimage.exposure.adjust_sigmoid",
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1)) # ZYX over each channel
        ),
        RescaleConfig(
            dtype=np.uint8, 
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1)) # ZYX over each channel
        ),
        FuncProcessorConfig(
            func="numpy.max", 
            kwargs={"axis":("C")}, # Project all channels into one
            strict_axis=False,     # don't throw exception if the input is a single channel image already
            dtype=np.uint16,
        ),
    ]
)
c_img, c_metadata = PreprocessFlow(ppcfg).run(img, metadata=metadata, workers=-1)
metadata, c_metadata

In [ ]:
stackview.slice(c_img)

## Segment lobes (3D, tiled MicroSAM)

`TiledSegmentationFlow` tiles the volume, runs MicroSAM per tile, merges overlapping detections (IoU / consensus thresholds), and applies `RegionFilter` constraints:

- **Cross-sectional area** ≥ 2000 in each orthogonal plane (`cross_sectional_area-xy`, `-xz`, `-yz`)
- **Aspect ratio** between 0.5 and 1.0

Embeddings are read from `embedding_path`. Adjust `tile_factor`, `resize_factor`, and filter ranges for your data scale.

In [ ]:
mscfg = MicroSAMSegmenterConfig(
    iterator_config=ArrayIteratorConfig(slice_def=()),
    embedding_path=embedding_path,
)

rfcfg = RegionFilterConfig(
    filters=[
        RangeFilterConfig(
            attribute="cross_sectional_area-xy", 
            range=(2000, np.inf)
        ),
        RangeFilterConfig(
            attribute="cross_sectional_area-xz", 
            range=(2000, np.inf)
        ),
        RangeFilterConfig(
            attribute="cross_sectional_area-yz", 
            range=(2000, np.inf)
        ),
        RangeFilterConfig(
            attribute="aspect_ratio", 
            range=(0.5, 1.0)
        ),
    ]
)

tsfcfg = TiledSegmentationFlowConfig(
    segmenter = mscfg,
    region_filter = rfcfg,
    tile_factor=(3,3),
    resize_factor=(0.25, 0.25),
    iou_threshold=0.5,
    consensus_threshold=0.75,
)
lobe_labels = TiledSegmentationFlow(tsfcfg).run(
    c_img, 
    metadata=c_metadata, 
    workers=2, 
    verbose=0, 
)

In [ ]:
print (f"Unique labels (incl. background): {np.unique(lobe_labels)}")

In [ ]:
vlabels = np.concatenate(len(metadata["channel_names"])*[lobe_labels], axis=-1)
stackview.blend(vimg.astype("uint16"), vlabels.astype("uint64"), blend_factor=40)

## Brain mask

A coarse **brain** label volume is derived from the lobe segmentation: any positive lobe label becomes foreground (`lobe_labels > 0`). Metadata is copied so the saved channel is named `Brain`.

In [ ]:
brain_label = (lobe_labels>0).astype("uint16")

b_metadata = copy.deepcopy(c_metadata)
b_metadata["channel_names"] = ["Brain"]

## Save segmentation

Writes two single-channel TIFF files next to the source basename:

- **Lobe** — integer label mask from `TiledSegmentationFlow`
- **Brain** — binary mask from the step above

Output path pattern: `{basename}-scene-{scene_index}.tif` (overwrites if present).

In [ ]:
c_metadata["channel_names"] = ["Lobe"]

b_metadata = copy.deepcopy(c_metadata)
b_metadata["channel_names"] = ["Brain"]

In [ ]:
# save lobe segmentation
imc = ImageWriterConfig(overwrite=True)
outpath = ".".join(path.split(".")[:-1]) + f"-scene-{scene_index}.tif"
ImageWriter(imc).run(lobe_labels, outpath, metadata=c_metadata)

# save brain segmentation
imc = ImageWriterConfig(overwrite=True)
outpath = ".".join(path.split(".")[:-1]) + f"-scene-{scene_index}.tif"
ImageWriter(imc).run(brain_label, outpath, metadata=b_metadata)


## Analyze lobe regions

`RegionAnalyzer` with `map_axes=True` produces a DataFrame of per-lobe measurements (volume, centroids, cross-sectional areas, bounding boxes, aspect ratio, slice annotations). Sort by `volume` to inspect the largest regions.

In [ ]:
racfg = RegionAnalyzerConfig(
    properties=["slice_annotations","volume", "centroid", "cross_sectional_area", "bbox", "aspect_ratio"],
    iterator_config = ArrayIteratorConfig(slice_def=()),
    output_type="dataframe",
    map_axes=True,
)
ra = RegionAnalyzer(racfg)

lobe_measurements = ra.run(lobe_labels, metadata=c_metadata)

In [ ]:
lobe_measurements.sort_values(["volume"],ascending=False)

## View in Napari (optional)

Interactive 3D viewer: lobe labels with measurement features, plus raw channels split with `unstack_image`. A synthetic row for label `0` (background) is prepended so Napari can attach features to every label layer.

In [ ]:
import napari
import pandas as pd

viewer = napari.Viewer()

In [ ]:
scale = metadata["physical_pixel_sizes"]
channel_colors = ("green", "blue", "red")
# img = np.expand_dims(img, axis=0)

# add fake measurement for background (label=0)
# this is needed for Napari
new_m = lobe_measurements.copy().reset_index()
background = pd.DataFrame({c: [0] if c=="label" else [np.nan] for c in new_m.columns.to_list()})
new_m = pd.concat([background, new_m], ignore_index=True)

# add lobe labels
viewer.add_labels(lobe_labels, name=c_metadata["channel_names"], features=new_m, scale=scale, opacity=0.5)

# split channels and add each channel as separate image layer
ch_images, ch_metadata = unstack_image(img, metadata=metadata, axis="C", strict=False)
for ch_meta, c_img, color in zip(ch_metadata, ch_images, channel_colors):
    viewer.add_image(c_img, name=f"{ch_meta['channel_names']}", scale=scale, colormap=color, blending="additive")
    
